In [ ]:
%pip install -q ultralytics==8.4.213 pillow-heif pandas matplotlib pyyaml
from google.colab import drive
from pathlib import Path
from PIL import Image
from pillow_heif import register_heif_opener
from ultralytics import YOLO
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import tempfile
import math
import json
import yaml
from datetime import datetime

register_heif_opener()
drive.mount('/content/drive')

Model locations

In [ ]:
BASE = Path('/content/drive/MyDrive/lecheros/Dataset')
FOLD = 1
RUNS = {
    'Original': BASE / 'bcs_training_v2' / f'classifier_fold_{FOLD}',
    'Dropout': BASE / 'bcs_classifier_dropout_v1' / f'classifier_fold_{FOLD}',
}
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
IMGSZ = 224
BATCH = 16
OUT = BASE / 'classifier_comparisons' / f"fold_{FOLD}_{datetime.now():%Y%m%d_%H%M%S}"

for name, run in RUNS.items():
    for filename in ['weights/best.pt', 'manifest.csv']:
        path = run / filename
        if not path.is_file():
            raise FileNotFoundError(f'{name}: missing {path}. Correct RUNS above.')
    print(name, run / 'weights/best.pt')
print('Device:', DEVICE)

Verify matching data and folds

In [ ]:
columns = ['image', 'label', 'source', 'cow_id', 'bcs', 'fold', 'split']
def normalize(frame):
    missing = set(columns) - set(frame.columns)
    if missing:
        raise ValueError(f'Manifest columns missing: {missing}')
    frame = frame[columns].copy()
    for col in ['image', 'label', 'source', 'cow_id', 'split']:
        frame[col] = frame[col].astype(str).str.strip()
    frame['bcs'] = pd.to_numeric(frame['bcs'], errors='raise')
    frame['fold'] = frame['fold'].astype(str).str.replace(r'\.0$', '', regex=True)
    if frame['image'].duplicated().any():
        raise ValueError('Duplicate images in manifest.')
    return frame.sort_values('image').reset_index(drop=True)

manifests = {
    name: normalize(pd.read_csv(run / 'manifest.csv', dtype={'cow_id': str}))
    for name, run in RUNS.items()
}
pd.testing.assert_frame_equal(
    manifests['Original'], manifests['Dropout'], check_dtype=False,
    obj='Original versus Dropout manifests',
)
manifest = manifests['Original']
ranch = manifest.loc[manifest['source'].eq('rancho')].copy()
ranch['animal_id'] = ranch['cow_id'].str.lower().str.replace(r'_r\d+$', '', regex=True)
if ranch.groupby('animal_id')['split'].nunique().gt(1).any():
    raise ValueError('The same physical cow appears in training and validation.')
expected_val = ranch['fold'].eq(str(FOLD))
if not (ranch['split'].eq('val') == expected_val).all():
    raise ValueError('Saved validation split does not match FOLD.')
validation = ranch.loc[ranch['split'].eq('val')].copy().reset_index(drop=True)
if validation.empty:
    raise ValueError('No ranch validation images found.')
for col in ['image', 'label']:
    missing_paths = [p for p in validation[col] if not Path(p).is_file()]
    if missing_paths:
        raise FileNotFoundError(f'Missing {col} files, e.g. {missing_paths[:5]}')
print('Same recorded dataset and splits confirmed.')
print('Validation photos:', len(validation))
print('Physical cows:', validation['animal_id'].nunique())
display(validation.groupby('bcs').size().rename('Photos').to_frame())

for name, run in RUNS.items():
    args_path = run / 'args.yaml'
    if args_path.is_file():
        args = yaml.safe_load(args_path.read_text()) or {}
        print(name, {k: args.get(k) for k in ['model', 'imgsz', 'dropout', 'freeze', 'seed']})
        if args.get('imgsz', IMGSZ) != IMGSZ:
            raise ValueError(f'{name}: training imgsz differs from IMGSZ.')

Rebuild validation crops

In [ ]:
def make_crop(image_path, label_path):
    lines = [line.split() for line in Path(label_path).read_text().splitlines() if line.strip()]
    if len(lines) != 1 or len(lines[0]) != 5:
        raise ValueError(f'Expected one YOLO box: {label_path}')
    values = np.asarray(lines[0], dtype=float)
    _, xc, yc, bw, bh = values
    if not np.isfinite(values).all() or not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
        raise ValueError(f'Invalid coordinates: {label_path}')
    if xc-bw/2 < -1e-5 or yc-bh/2 < -1e-5 or xc+bw/2 > 1+1e-5 or yc+bh/2 > 1+1e-5:
        raise ValueError(f'Box outside image: {label_path}')
    with Image.open(image_path) as original:
        image = original.convert('RGB')  # Same orientation handling as training.
    w, h = image.size
    box = (max(0, math.floor((xc-bw/2)*w)), max(0, math.floor((yc-bh/2)*h)),
           min(w, math.ceil((xc+bw/2)*w)), min(h, math.ceil((yc+bh/2)*h)))
    crop = image.crop(box)
    side = max(crop.size)
    canvas = Image.new('RGB', (side, side), (114, 114, 114))
    canvas.paste(crop, ((side-crop.width)//2, (side-crop.height)//2))
    return canvas

crop_dir = Path(tempfile.mkdtemp(prefix='bcs_comparison_'))
crop_paths = []
for i, row in validation.iterrows():
    path = crop_dir / f'image_{i:06d}.jpg'
    make_crop(row['image'], row['label']).save(path)
    crop_paths.append(str(path))
print('Prepared identical crops for both models:', len(crop_paths))
fig, axes = plt.subplots(1, min(4, len(crop_paths)), figsize=(12, 4), squeeze=False)
for i, ax in enumerate(axes[0]):
    with Image.open(crop_paths[i]) as im:
        ax.imshow(im)
    ax.set_title(f"{validation.iloc[i]['cow_id']} | BCS {validation.iloc[i]['bcs']:.2f}")
    ax.axis('off')
plt.tight_layout()
plt.show()

Predict with both saved classifiers

In [ ]:
comparison = validation[['image', 'cow_id', 'animal_id', 'bcs']].copy()
class_maps = {}
for name, run in RUNS.items():
    model = YOLO(str(run / 'weights/best.pt'))
    if model.task != 'classify':
        raise ValueError(f'{name} is not a classifier.')
    class_maps[name] = {int(k): str(v) for k, v in model.names.items()}
    available_scores = {float(v) for v in model.names.values()}
    missing_scores = set(comparison['bcs']) - available_scores
    if missing_scores:
        raise ValueError(f'{name} cannot predict these validation classes: {missing_scores}')
    predictions = []
    for start in range(0, len(crop_paths), BATCH):
        results = model.predict(crop_paths[start:start+BATCH], imgsz=IMGSZ,
                                device=DEVICE, augment=False, verbose=False)
        for result in results:
            if result.probs is None:
                raise ValueError('Missing classification output.')
            predictions.append(float(result.names[result.probs.top1]))
    if len(predictions) != len(comparison) or not np.isfinite(predictions).all():
        raise ValueError(f'{name}: incomplete or invalid predictions.')
    comparison[name] = predictions
    comparison[f'{name}_absolute_error'] = np.abs(comparison[name] - comparison['bcs'])
    print(f'{name}: {len(predictions)} predictions')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
if set(class_maps['Original'].values()) != set(class_maps['Dropout'].values()):
    print('NOTE: the models have different class sets; this is not a dropout-only comparison.')

Compare metrics and success criteria

In [ ]:
def metrics(frame, name):
    err = frame[name].to_numpy() - frame['bcs'].to_numpy()
    ae = np.abs(err)
    return {'Model': name, 'Photos': len(frame), 'Coverage (%)': 100.0,
            'MAE': ae.mean(), 'RMSE': np.sqrt(np.mean(err**2)),
            'Within ±0.25 (%)': 100*np.mean(ae <= .25+1e-9),
            'Exact accuracy (%)': 100*np.mean(np.isclose(err, 0, atol=1e-9, rtol=0))}

scopes = {'All BCS': comparison, 'BCS 3.25–3.50': comparison.loc[comparison['bcs'].between(3.25, 3.50)]}
metric_names = ['MAE', 'RMSE', 'Within ±0.25 (%)', 'Exact accuracy (%)']
summary_rows, change_rows, criteria_rows = [], [], []
for scope, frame in scopes.items():
    print('\n' + scope)
    if frame.empty:
        print('No photos in this interval.')
        continue
    table = pd.DataFrame([metrics(frame, name) for name in RUNS]).set_index('Model')
    display(table.round(4))
    summary_rows.extend([{'Scope': scope, **metrics(frame, name)} for name in RUNS])
    local_changes = []
    for metric in metric_names:
        delta = float(table.loc['Dropout', metric] - table.loc['Original', metric])
        improved = delta < 0 if metric in ['MAE', 'RMSE'] else delta > 0
        result = 'Unchanged' if np.isclose(delta, 0, atol=1e-9) else ('Improved' if improved else 'Worse')
        row = {'Scope': scope, 'Metric': metric, 'Change': delta, 'Result': result}
        local_changes.append(row)
        change_rows.append(row)
    display(pd.DataFrame(local_changes).drop(columns='Scope').round(4))
    old = frame['Original_absolute_error'].to_numpy()
    new = frame['Dropout_absolute_error'].to_numpy()
    print(f'Photos: {(new < old-1e-9).sum()} improved; {np.isclose(new, old).sum()} unchanged; {(new > old+1e-9).sum()} worsened.')
    if scope == 'BCS 3.25–3.50':
        for name, row in table.iterrows():
            checks = {'MAE ≤ 0.25': bool(row['MAE'] <= .25+1e-9),
                      'RMSE ≤ 0.35': bool(row['RMSE'] <= .35+1e-9),
                      'Within ±0.25 ≥ 80%': bool(row['Within ±0.25 (%)'] >= 80-1e-9)}
            criteria_rows.append({'Model': name, **checks, 'All criteria met': all(checks.values())})
        display(pd.DataFrame(criteria_rows).replace({True: 'PASS', False: 'FAIL'}))
summary = pd.DataFrame(summary_rows)
changes = pd.DataFrame(change_rows)

Save comparison results to Drive

In [ ]:
OUT.mkdir(parents=True, exist_ok=False)
comparison.to_csv(OUT / 'predictions.csv', index=False)
summary.to_csv(OUT / 'metrics.csv', index=False)
changes.to_csv(OUT / 'metric_changes.csv', index=False)
pd.DataFrame(criteria_rows).to_csv(OUT / 'success_criteria.csv', index=False)
(OUT / 'comparison_config.json').write_text(json.dumps({
    'fold': FOLD, 'runs': {k: str(v) for k, v in RUNS.items()},
    'imgsz': IMGSZ, 'class_maps': class_maps,
    'input': 'Manual annotation crops, square padded; no detector',
}, indent=2))
print('Saved comparison:', OUT)
print('Compare metrics.csv and metric_changes.csv; inspect predictions.csv for individual errors.')